In [6]:
# -----------------------------------------------------------------
# 1. Import Libraries
# -----------------------------------------------------------------
import pandas as pd
import mysql.connector

# -----------------------------------------------------------------
# 2. Load dataset
# -----------------------------------------------------------------
df = pd.read_csv('customer_shopping_behavior.csv')

# -----------------------------------------------------------------
# 3. EDA
# -----------------------------------------------------------------
df.head()
df.info()
df.describe()
df.isnull().sum()

df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df = df.rename(columns = {'purchase_amount_(usd)' : 'purchase_amount'})

df.columns

# create column age_group
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

# create column purchase_frequency_days
df['frequency_of_purchases'] = df['frequency_of_purchases'].replace({
        'Every 3 Months':'Quarterly',
        'Bi-Weekly':'Fortnightly',
        'Annually': 'Yearly',
})

frequency_mapping = {
    'Weekly' : 7,
    'Fortnightly' : 14,
    'Monthly' : 30,
    'Quarterly' : 90,
    'Yearly' : 365
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
df[['purchase_frequency_days', 'frequency_of_purchases']].head(10)


df[['discount_applied','promo_code_used']].head(10)
(df['discount_applied'] == df['promo_code_used']).all()

df.drop(columns=['promo_code_used'], inplace=True)

df.to_csv('customer_shopping_behavior_cleaned.csv', index=False)

# Read cleaned data
df = pd.read_csv("customer_shopping_behavior_cleaned.csv")

# -----------------------------------------------------------------
# 4. Connect to MySQL
# -----------------------------------------------------------------
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="fafoks616061",
    database="customer_behavior"
)

cursor = conn.cursor()

# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
        customer_id INT,
        age INT,
        gender VARCHAR(50),
        item_purchased VARCHAR(50),
        category VARCHAR(50),
        purchase_amount INT,
        location VARCHAR(50),
        size VARCHAR(50),
        color VARCHAR(50),
        season VARCHAR(50),
        review_rating FLOAT,
        subscription_status VARCHAR(50),
        shipping_type VARCHAR(50),
        discount_applied VARCHAR(50),
        previous_purchases INT,
        payment_method VARCHAR(50),
        frequency_of_purchases VARCHAR(50),
        age_group VARCHAR(50),
        purchase_frequency_days INT
)
""")

# Insert data
for _, row in df.iterrows():
    cursor.execute("""
        INSERT INTO customers (customer_id, age, gender, item_purchased, category, purchase_amount, location, size, color, season, review_rating, subscription_status, shipping_type, discount_applied, previous_purchases, payment_method, frequency_of_purchases, age_group, purchase_frequency_days)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, tuple(row))

# Commit and close
conn.commit()
cursor.close()
conn.close()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   